Obtención de matrículas a partir del dataset (ejecutar si no existe el directorio "plates/")

In [ ]:
import json
import cv2
import numpy as np
from pathlib import Path

DATASET = Path("dataset")
OUT = Path("plates")
MIN_W = 60
PAD = 0.05
IMG_EXT = {".jpg", ".jpeg", ".png"}

def leer(p):
    return cv2.imdecode(np.fromfile(str(p), np.uint8), cv2.IMREAD_COLOR)

def guardar(p, img):
    cv2.imencode(".jpg", img)[1].tofile(str(p))

for split in ["train", "val", "test"]:
    (OUT / split).mkdir(parents=True, exist_ok=True)

    imagenes = {p.stem: p for p in (DATASET / split / "images").iterdir()
                if p.suffix.lower() in IMG_EXT}
    jsons = {p.stem: p for p in (DATASET / split / "labels").glob("*.json")}
    parejas = sorted(set(imagenes) & set(jsons))

    validas = []   # (nombre, ruta_imagen, caja, texto)
    for nombre in parejas:
        for i, obj in enumerate(json.load(open(jsons[nombre], encoding="utf-8"))):
            texto = str(obj.get("plate") or "").replace(" ", "").upper()
            if texto and texto != "NONE":
                validas.append((f"{nombre}_{i}", imagenes[nombre], obj["rectangle"], texto))

    lineas, pequenas = [], 0
    for nombre, ruta, (cx, cy, w, h), texto in validas:
        img = leer(ruta)
        H, W = img.shape[:2]
        x1 = max(int((cx - w / 2 - w * PAD) * W), 0)
        x2 = min(int((cx + w / 2 + w * PAD) * W), W)
        y1 = max(int((cy - h / 2 - h * PAD) * H), 0)
        y2 = min(int((cy + h / 2 + h * PAD) * H), H)
        if x2 - x1 < MIN_W:
            pequenas += 1
            continue
        guardar(OUT / split / f"{nombre}.jpg", img[y1:y2, x1:x2])
        lineas.append(f"{split}/{nombre}.jpg\t{texto}")

    (OUT / f"{split}.txt").write_text("\n".join(lineas), encoding="utf-8")
    print(f"{split}: imágenes {len(imagenes)}, jsons {len(jsons)}, parejas {len(parejas)}, "
          f"con matrícula {len(validas)}, descartadas por ancho {pequenas}, "
          f"finales {len(lineas)}")

Importación de paquetes

In [1]:
import os
import subprocess
import sys
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import csv
import math
import yaml
import paddle
import cv2
import numpy as np
from collections import defaultdict
from paddleocr import PaddleOCR
from pathlib import Path

sys.path.insert(0, "PaddleOCR")
from ppocr.modeling.architectures import build_model
from ppocr.postprocess import build_post_process
from ppocr.utils.save_load import load_pretrained_params


c:\Users\Dylan\anaconda3\envs\P4_paddle\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Skipping import of the encryption module.


Entrenamiento lector de matrículas

In [2]:
DATA_DIR = "plates"
EPOCHS = 50
BATCH_SIZE = 8
LR = 1e-4
OUT_DIR = "results/plate-ocr"
CONFIG = "PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml"
PRETRAINED = ("https://paddle-model-ecology.bj.bcebos.com/paddlex/"
              "official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams")

def correr(cmd):
    """Ejecuta el comando y muestra su salida (stdout+stderr) en el notebook."""
    prefix = os.path.dirname(sys.executable)   # carpeta del entorno VC_P4
    env = os.environ.copy()
    env["PATH"] = os.pathsep.join([
        prefix,
        os.path.join(prefix, "Library", "bin"),
        os.path.join(prefix, "Scripts"),
        env["PATH"],
    ])
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace", env=env)
    for linea in p.stdout:
        print(linea, end="")
    if p.wait() != 0:
        raise RuntimeError(f"El comando terminó con código {p.returncode}")

# Diccionario de caracteres (matrículas: dígitos y letras)
DICT_PATH = f"{DATA_DIR}/dict.txt"
with open(DICT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"))

# Clonar PaddleOCR (solo la primera vez)
if not os.path.exists("PaddleOCR"):
    correr(["git", "clone", "--depth", "1", "https://github.com/PaddlePaddle/PaddleOCR.git"])
    correr([sys.executable, "-m", "pip", "install", "-r", "PaddleOCR/requirements.txt"])

# Entrenar
overrides = [
    f"Global.epoch_num={EPOCHS}",
    f"Global.save_model_dir={OUT_DIR}",
    f"Global.pretrained_model={PRETRAINED}",
    f"Global.character_dict_path={DICT_PATH}",
    "Global.use_space_char=False",
    "Global.eval_batch_step=[0,100]",
    f"Optimizer.lr.learning_rate={LR}",
    f"Train.dataset.data_dir={DATA_DIR}",
    f"Train.dataset.label_file_list=['{DATA_DIR}/train.txt']",
    f"Train.loader.batch_size_per_card={BATCH_SIZE}",
    f"Train.sampler.first_bs={BATCH_SIZE}",
    "Train.loader.num_workers=0",
    f"Eval.dataset.data_dir={DATA_DIR}",
    f"Eval.dataset.label_file_list=['{DATA_DIR}/val.txt']",
    f"Eval.loader.batch_size_per_card={BATCH_SIZE}",
    "Eval.loader.num_workers=0",
]
correr([sys.executable, "-u", "PaddleOCR/tools/train.py", "-c", CONFIG, "-o", *overrides])

# Exportar el mejor modelo
correr([sys.executable, "PaddleOCR/tools/export_model.py", "-c", CONFIG, "-o",
        f"Global.pretrained_model={OUT_DIR}/best_accuracy",
        f"Global.save_inference_dir={OUT_DIR}/inference",
        f"Global.character_dict_path={DICT_PATH}"])

print("Listo. Modelo en", f"{OUT_DIR}/inference")

Skipping import of the encryption module.
[2026/11/10 18:03:49] ppocr INFO: Architecture : 
[2026/11/10 18:03:49] ppocr INFO:     Backbone : 
[2026/11/10 18:03:49] ppocr INFO:         name : PPLCNetV3
[2026/11/10 18:03:49] ppocr INFO:         scale : 0.95
[2026/11/10 18:03:49] ppocr INFO:     Head : 
[2026/11/10 18:03:49] ppocr INFO:         head_list : 
[2026/11/10 18:03:49] ppocr INFO:             CTCHead : 
[2026/11/10 18:03:49] ppocr INFO:                 Head : 
[2026/11/10 18:03:49] ppocr INFO:                     fc_decay : 1e-05
[2026/11/10 18:03:49] ppocr INFO:                 Neck : 
[2026/11/10 18:03:49] ppocr INFO:                     depth : 2
[2026/11/10 18:03:49] ppocr INFO:                     dims : 120
[2026/11/10 18:03:49] ppocr INFO:                     hidden_dims : 120
[2026/11/10 18:03:49] ppocr INFO:                     kernel_size : [1, 3]
[2026/11/10 18:03:49] ppocr INFO:                     name : svtr
[2026/11/10 18:03:49] ppocr INFO:                     use

Prueba del modelo PaddleOCR

In [3]:
OUT_DIR = Path("results") / "plate-ocr"
VIDEO_ORIG = "video_test.MP4"
CSV_IN = OUT_DIR / "tracking.csv"
VIDEO_OUT = OUT_DIR / "video_final.mp4"
CSV_OUT = OUT_DIR / "tracking_final.csv"
DICT_PATH = "plates/dict.txt"
OCR_CKPT = "results/plate-ocr/best_model/model"
OCR_CONFIG = "results/plate-ocr/config.yml"
IMG_C, IMG_H, IMG_W = 3, 48, 320
PAD = 0.05

# --- Cargar el lector entrenado ---
with open(OCR_CONFIG, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg["Global"]["character_dict_path"] = DICT_PATH
post = build_post_process(cfg["PostProcess"], cfg["Global"])
char_num = len(post.character)
cfg["Architecture"]["Head"]["out_channels_list"] = {
    "CTCLabelDecode": char_num,
    "SARLabelDecode": char_num + 2,
    "NRTRLabelDecode": char_num + 3,
}
reader = build_model(cfg["Architecture"])
load_pretrained_params(reader, OCR_CKPT)
reader.eval()

def leer_matricula(crop):
    """Redimensiona manteniendo proporción (48x320), normaliza y decodifica con CTC."""
    h, w = crop.shape[:2]
    nw = min(int(math.ceil(IMG_H * w / h)), IMG_W)
    img = cv2.resize(crop, (nw, IMG_H)).astype("float32")
    img = (img.transpose(2, 0, 1) / 255 - 0.5) / 0.5
    pad = np.zeros((IMG_C, IMG_H, IMG_W), dtype="float32")
    pad[:, :, :nw] = img
    with paddle.no_grad():
        preds = reader(paddle.to_tensor(pad[None]))
    if isinstance(preds, dict):
        preds = preds["ctc"]
    texto, _ = post(preds.numpy())[0]
    return texto

def poner_texto(img, texto, x, y):
    """Texto verde sobre fondo negro para que se lea bien."""
    (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
    y = max(y, th + 6)
    cv2.rectangle(img, (x, y - th - 6), (x + tw + 4, y + 2), (0, 0, 0), -1)
    cv2.putText(img, texto, (x + 2, y - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

# --- Leer el CSV de YOLO y agrupar filas por fotograma ---
with open(CSV_IN, newline="", encoding="utf-8") as f:
    filas = list(csv.DictReader(f))
por_frame = defaultdict(list)
for fila in filas:
    por_frame[int(fila["fotograma"])].append(fila)

cap = cv2.VideoCapture(VIDEO_ORIG)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir {VIDEO_ORIG}")
fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(str(VIDEO_OUT), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

n = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    for fila in por_frame.get(n, []):
        # Caja del objeto
        x1, y1, x2, y2 = (int(fila[k]) for k in ("x1", "y1", "x2", "y2"))
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 128, 0), 2)
        cv2.putText(frame, f"{fila['tipo_objeto']} #{fila['id_tracking']}", (x1, max(y1 - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 128, 0), 2)

        if not fila["mx1"]:
            continue
        mx1, my1, mx2, my2 = (int(fila[k]) for k in ("mx1", "my1", "mx2", "my2"))
        bw, bh = mx2 - mx1, my2 - my1
        # El recorte se toma antes de dibujar la caja de la matrícula
        crop = frame[max(my1 - int(bh * PAD), 0):min(my2 + int(bh * PAD), H),
                     max(mx1 - int(bw * PAD), 0):min(mx2 + int(bw * PAD), W)].copy()
        cv2.rectangle(frame, (mx1, my1), (mx2, my2), (0, 255, 0), 2)
        if crop.shape[1] >= 20 and crop.shape[0] >= 8:
            fila["texto_matricula"] = leer_matricula(crop)
            poner_texto(frame, fila["texto_matricula"], mx1, my1 - 6)

    writer.write(frame)
    n += 1

cap.release()
writer.release()

with open(CSV_OUT, "w", newline="", encoding="utf-8") as f:
    wr = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
    wr.writeheader()
    wr.writerows(filas)

print("Vídeo:", VIDEO_OUT)
print("CSV final:", CSV_OUT)

[2026/11/11 13:53:17] ppocr INFO: load pretrain successful from results/plate-ocr/best_model/model
Vídeo: results\plate-ocr\video_final.mp4
CSV final: results\plate-ocr\tracking_final.csv
